# Onboarding Agent Interview Guide - LangChain

This guide is tailored for your standalone LangChain onboarding app and broader LangChain interviews.

## 1. Quick Project Pitch

### Q1. What did you build with LangChain in this project?
A. I built a FastAPI onboarding assistant that uses LangChain tool-calling patterns to guide users through required document workflows. The assistant fetches required documents and required fields through controlled tools, extracts candidate fields from uploaded files, and enforces human confirmation before save.

### Q2. Why use LangChain here?
A. LangChain gave me a structured way to define tools, prompts, and model invocation logic. It helped separate conversational intelligence from deterministic backend operations like validation, parsing, and persistence.

### Q3. What is the business value?
A. It reduces manual onboarding effort, improves data consistency, and speeds document processing while keeping a human-in-the-loop confirmation checkpoint.

### Q4. How did you keep it production-oriented?
A. I added guardrails, upload checks, tool boundaries, required-field validation, and deterministic save behavior. I also designed service boundaries for ASP.NET integration and Bedrock fallback through stub-first architecture.

## 2. Architecture and Flow

### Q5. Explain the end-to-end flow.
A.
1. Create session.
2. Fetch required documents.
3. Fetch required fields for selected document.
4. Upload file and parse text.
5. AI assists with extraction workflow.
6. User confirms or edits fields.
7. Confirm endpoint validates and saves.

### Q6. What are the main modules in the LangChain app?
A.
1. main.py: API endpoints.
2. services/agent_service.py: LangChain orchestration.
3. services/asp_service.py: external system integration.
4. services/document_parser.py: file parsing and extraction hints.
5. guardrails.py: safety controls.
6. session_store.py: session state.

### Q7. Where is LangChain used exactly?
A. In agent_service.py using prompt templates, tool definitions, and agent execution flow. In non-stub mode, the service can run tool-calling with Bedrock-backed chat.

### Q8. What is deterministic vs non-deterministic in your design?
A. Conversational responses are LLM-assisted. Required-field checks, upload validation, ownership checks, and save payload construction are deterministic.

## 3. LangChain Fundamentals

### Q9. What are core LangChain primitives?
A. Prompts, LLM wrappers, tools, output parsers, memory abstractions, chains, and agents.

### Q10. Difference between chain and agent?
A. A chain is predefined execution order. An agent dynamically decides next action, often selecting tools at runtime.

### Q11. Why tool-calling instead of plain chat completion?
A. Tool-calling lets the model ask for authoritative backend data at runtime instead of hallucinating required documents or rules.

### Q12. What is StructuredTool and why use it?
A. StructuredTool lets you define typed tool interfaces, making tool invocation safer and clearer for both code and model behavior.

### Q13. How do you control agent behavior?
A. Prompt constraints, strict tool set, deterministic post-validation, response limits, and retry/fallback handling.

### Q14. What is prompt templating strategy?
A. I use a system directive for scope and safety, then pass user input through templated human messages.

### Q15. How do you avoid uncontrolled tool use?
A. Only expose allow-listed tools and keep critical side effects outside agent control.

## 4. Project-Specific LangChain Deep Dive

### Q16. Which tools did your agent use?
A.
1. get_required_documents.
2. get_required_fields.

### Q17. Why not allow save_document as a tool?
A. Save is high-risk and must remain deterministic after explicit user confirmation and backend validation.

### Q18. How do you persist context between turns?
A. Session history and session state are persisted in memory per session_id.

### Q19. How do you handle stale data?
A. Required fields are re-fetched on confirm, not trusted solely from previous turns.

### Q20. How do you prevent wrong document acceptance?
A. Expected document is tracked from user selection and checked against extraction outputs and required fields.

### Q21. How do you handle parser limitations?
A. Parser extracts hints with confidence; user confirms final values. Low-confidence values are surfaced for manual correction.

### Q22. Why combine endpoint-driven flow with chat?
A. It gives predictable UX for required steps while still allowing conversational guidance.

## 5. Security and Guardrails

### Q23. What guardrails are implemented?
A.
1. Prompt injection phrase detection.
2. Off-topic detection and short-circuit response.
3. File extension allowlist.
4. File size limit.
5. Magic-byte signature verification.
6. Confirm-field whitelist and length constraints.

### Q24. Why magic-byte validation?
A. Extension can be spoofed. Magic bytes verify actual file type.

### Q25. How do you prevent prompt hijack?
A. Reject requests containing instruction override patterns and keep system prompt constraints strict.

### Q26. How do you reduce data leakage risk?
A. Keep file data in memory only for active flow and clear temp payload after successful save.

### Q27. How would you improve security further?
A.
1. JWT claim verification and session ownership enforcement.
2. PII masking in logs.
3. AV/malware scanning.
4. Signed upload URLs and encryption controls.

### Q28. How do you defend against off-topic abuse?
A. Off-topic queries are detected and redirected without unnecessary model calls.

## 6. Cost Optimization

### Q29. What cost controls are in place?
A.
1. Off-topic short-circuiting.
2. Stub mode for local development.
3. Tool-driven grounding to reduce long exploratory prompts.
4. Human-confirmation gate to avoid repeated expensive save retries.

### Q30. How would you optimize Bedrock cost further?
A.
1. Use smaller model for simple intents.
2. Route complex extraction to larger model only when needed.
3. Add token budgeting and response length caps.
4. Cache required document list per member and TTL.
5. Compact historical context.

### Q31. How do you decide when to call the LLM?
A. Only when conversational value is needed. Deterministic endpoints handle structural steps.

### Q32. Would RAG reduce cost in this project?
A. Limited impact for core extraction flow, but useful for policy FAQ answers so the model does not infer policy content.

## 7. Reliability and Observability

### Q33. What should be logged?
A.
1. Request ID.
2. Session ID.
3. Endpoint latency.
4. Tool calls and outcomes.
5. Validation failures.
6. Save success/failure states.

### Q34. What should not be logged?
A. Raw PII values, full document text, secrets, and token content.

### Q35. How do you handle transient external failures?
A. Retries with jitter, circuit breaking, graceful user message, and idempotent confirmation paths.

### Q36. How to avoid duplicate saves?
A. Idempotency key based on session_id + document_id + content hash.

## 8. Agent Evaluation

### Q37. How do you evaluate a LangChain agent quality?
A.
1. Task success rate.
2. Tool selection precision.
3. Hallucination rate.
4. User edit rate after extraction.
5. Confirm-to-save completion rate.
6. Latency and token cost.

### Q38. What offline evaluation set would you build?
A. Curated document samples across formats, quality levels, and edge cases with expected extraction labels.

### Q39. What online metrics matter most?
A. Abandonment rate, retries, time-to-completion, field correction rate, and save failure rate.

### Q40. How to evaluate guardrails?
A. Maintain red-team prompts and malicious file fixtures; track block precision and false positives.

### Q41. How do you evaluate tool-use correctness?
A. Compare intended tool invocation sequence vs observed sequence and measure unnecessary tool calls.

## 9. Scenario-Based Q and A

### Q42. User asks joke/weather while onboarding.
A. Return scoped redirect immediately without model call.

### Q43. User uploads exe renamed to pdf.
A. Magic-byte check fails and upload is rejected.

### Q44. Required fields API is down.
A. Return retryable error, keep session alive, and do not attempt save.

### Q45. Save succeeds but response parsing fails.
A. Treat as uncertain outcome, query save status endpoint or reconcile via idempotency key.

### Q46. Model suggests values not in document.
A. Mark low confidence and require user confirmation before save.

### Q47. Two rapid confirm calls for same document.
A. Use lock/idempotency to ensure single save side effect.

### Q48. Large image increases latency.
A. Downscale image before OCR/model call and enforce size thresholds.

### Q49. Agent repeatedly calls same tool.
A. Enforce max tool iterations and return fallback deterministic guidance.

### Q50. User switches document mid-flow.
A. Clear selected_document_id-specific temp state and require required-fields refresh.

## 10. LangChain Outside Project

### Q51. What are Runnable interfaces in LangChain?
A. They standardize compose/invoke/stream behavior across prompts, models, and chains.

### Q52. What is LCEL and why useful?
A. LangChain Expression Language allows composable pipelines with clear data flow and better readability.

### Q53. What is memory in modern LangChain?
A. Memory is generally explicit state handling or message history wrappers rather than hidden implicit memory.

### Q54. When to use LangServe?
A. When exposing chains/agents as deployable APIs with built-in tracing and schema endpoints.

### Q55. How does LangSmith help?
A. Trace-level observability for prompts, tool calls, latency, and debugging agent behavior.

### Q56. Difference between retrieval QA and agentic tool use?
A. Retrieval QA focuses on grounding from documents; agentic tool use focuses on action planning and dynamic function execution.

### Q57. How to tune prompts safely?
A. Keep system role minimal and strict, isolate templates, version prompts, and evaluate before release.

### Q58. How to reduce hallucinations in LangChain apps?
A.
1. Ground with tools/RAG.
2. Structured output schemas.
3. Explicit unknown handling.
4. Deterministic validation layers.

## 11. Security and Compliance Extras

### Q59. How would you handle PII compliance?
A. Data minimization, masking, strict retention windows, encrypted transport/storage, audit logs, and role-based access.

### Q60. What is least privilege here?
A. Bedrock and backend service roles should have only required API permissions.

### Q61. How to secure secrets?
A. Use AWS Secrets Manager or SSM, not source control.

### Q62. How to harden uploads?
A. Add MIME sniffing, AV scanning, sandbox conversion, and quarantine pipeline.

## 12. Resume and Leadership Style Questions

### Q63. What was your key technical decision?
A. Keeping save operations deterministic and outside agent autonomy while still using LangChain for guided interaction.

### Q64. What trade-off did you make?
A. Slightly more backend orchestration complexity in exchange for reliability, compliance, and auditability.

### Q65. What would you improve next?
A.
1. Redis session store.
2. LangSmith tracing.
3. Automated eval harness.
4. Multi-model routing.
5. Strong JWT auth and ownership checks.

### Q66. Why is this a good enterprise AI example?
A. It combines practical LLM usage with strict operational controls, security boundaries, and measurable business outcomes.

## 13. Rapid-Fire One-Liners

### Q67. LangChain in one line?
A. A framework to build LLM applications with structured prompts, tools, and orchestration.

### Q68. Biggest risk in agentic systems?
A. Unbounded autonomy without deterministic guardrails.

### Q69. Best mitigation?
A. Tight scope, tool allowlist, validation gates, and human confirmation for sensitive actions.

### Q70. Key success metric for this project?
A. High completion rate with low correction rate and low cost per completed onboarding.
